# Healthcare Agent Learning Lab 6/10

## Uncertainty and abstention

**Suggested time:** 25 minutes  
**Demonstration type:** Sampling simulation or live repeated model calls

### Learning objectives

- Distinguish answer consistency from correctness and calibration.
- Relate an abstention threshold to escalation burden and residual risk.
- Identify questions that are answerable from the record versus clinically ambiguous.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.

> **Workshop note:** The complete live notebook intentionally makes 76 model calls. Its elapsed time and cost are part of the lesson about the operational burden of repeated-sampling uncertainty methods.


## Demonstration
**Goal:** make an agent **know when it doesn't know**, and *abstain / escalate* instead of answering confidently.

We estimate confidence with **self-consistency**: ask the model the same thing several times (temperature > 0) and measure how much the answers agree. High agreement → answer; low agreement → hand off to a human. Every question is grounded in a short record, so a real model is reasoning over given facts — not inventing them. (Runs in MOCK mode with no API key; set a key to make it real.)


> ### In plain terms
> A trustworthy agent has to **know when it doesn't know.** The trick used here: ask the model the same question several times. If the answers **agree**, it's confident; if they **scatter**, hand the question to a human instead of guessing.
>
> Two important details: we compare answers by **meaning, not exact wording** (so "warfarin overdose" and "warfarin-related bleeding" count as the same idea), and every question is asked against one specific patient record.
>
> **What you'll see:** first a confident question vs. an uncertain one; then a table where you can watch the "confidence bar" (tau) move and turn answers into escalations. One honest caveat throughout: **agreement means the model is consistent, not that it's right.**


In [ ]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then set env vars. OpenRouter example:
#        export OPENAI_BASE_URL=https://openrouter.ai/api/v1
#        export OPENAI_API_KEY=sk-or-...          # your OpenRouter key (never commit it)
#        export MODEL=openai/gpt-4o-mini          # any OpenRouter model id
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

import random
_RNG = random.Random(7)
# The model's latent answer spread per question keyword. In a REAL run you don't write
# these — you get the spread for free by sampling at temperature>0. Here they simulate it.
_DIST = {
    "creatinine":    {"1.2 mg/dL": 0.95, "1.9 mg/dL": 0.05},
    "potassium":     {"4.1 mmol/L": 0.85, "5.6 mmol/L": 0.15},
    "warfarin dose": {"5 mg": 0.75, "7.5 mg": 0.25},
    "chest pain":    {"stable angina": 0.45, "GERD": 0.3, "anxiety": 0.25},   # genuinely split
    "cause of the":  {"sepsis": 0.55, "PE": 0.25, "dehydration": 0.20},       # borderline
}
def _mock(messages, temperature=0):
    u = " ".join(m["content"] for m in messages if m["role"] == "user").lower()
    u = u.split("question:")[-1]          # match on the QUESTION, not the record above it
    for key, dist in _DIST.items():
        if key in u:
            return _RNG.choices(list(dist), weights=list(dist.values()))[0]
    return "unsure"


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### A record, and confidence by agreement
Factual questions are answerable **from the record** (the model should be confident and right); diagnostic questions genuinely are not (it should be unsure). Confidence = the fraction of samples that agree with the majority answer.


In [ ]:
from collections import Counter
import re

RECORD = ("58M, post-op day 5 after DVT. Labs today: creatinine 1.2 mg/dL, potassium 4.1 mmol/L, "
          "INR 4.2 (range 2.0-3.0). Medications: warfarin 5 mg daily, lisinopril 10 mg daily. "
          "Now reports intermittent chest pain with a nonspecific ECG and one episode of transient hypotension.")
SYS = ("You are a clinical assistant. Use ONLY the record provided. "
       "Answer with a short value or phrase only — no explanation.")

def ask_n(question, n=10):
    "Sample the model n times (temperature 1) on the same record + question."
    msgs = [{"role":"system","content":SYS},
            {"role":"user","content":f"Record:\n{RECORD}\n\nQuestion: {question}"}]
    return [chat(msgs, temperature=1) for _ in range(n)]

# --- Agreement by MEANING, not exact wording --------------------------------
# A real model phrases the same idea many ways ("warfarin overdose",
# "warfarin-related bleeding"). We map each answer to a canonical bucket so
# "agreement" measures whether the model MEANS the same thing, not whether it
# typed the same characters.
_BUCKETS = [
 ("cardiac / ischemia",         ["angina","cardiac","ischem","coronary","acs","myocard"]),
 ("anticoagulation / bleeding", ["warfarin","anticoag","bleed","coagulopath","inr","supratherap"]),
 ("GERD / reflux",              ["gerd","reflux","heartburn","esophag"]),
 ("anxiety / panic",            ["anxiety","panic","psychogenic"]),
 ("pulmonary embolism",         ["pulmonary embol","embolism"]),
 ("sepsis / infection",         ["sepsis","septic","infection"]),
 ("hypovolemia / dehydration",  ["dehydr","hypovol","volume depl"]),
 ("musculoskeletal",            ["musculoskelet","costochondr","muscle strain"]),
]
def canon(s):
    "Normalise a model answer to a comparable form (a value, or a clinical bucket)."
    s = s.strip().lower().rstrip(".")
    v = re.fullmatch(r"(?:the\s+)?(\d+(?:\.\d+)?)\s*(mg/dl|mmol/l|mg|mcg|units?|%)?", s)
    if v:                                    # a value answer, e.g. "1.2 mg/dl"
        return f"{float(v.group(1)):g} {(v.group(2) or '').replace(' ','')}".strip()
    for label, kws in _BUCKETS:              # a phrase answer -> clinical bucket
        if any(k in s for k in kws): return label
    return s
def matches(ans, correct):
    a, c = canon(ans), canon(correct)
    return c in a or a in c

def confidence(question, n=10):
    "Sample, group by meaning, return (top bucket, agreement fraction, raw samples)."
    raw = ask_n(question, n)
    buckets = [canon(s) for s in raw]
    top, cnt = Counter(buckets).most_common(1)[0]
    return top, cnt/len(buckets), raw


### One confident question, one genuinely uncertain one
Same machinery, two very different confidence signals. The creatinine is written in the record, so the model should agree with itself every time. The cause of the transient hypotension is *not* in the record, so honest sampling should scatter — and low agreement should route it to a human.


In [ ]:
# Two questions: one answerable FROM THE RECORD, one that genuinely is not.
for q in ["What is the patient's most recent creatinine? (value only)",
          "What is the single most likely cause of the transient hypotension? (one phrase)"]:
    top, conf, raw = confidence(q, n=8)
    print("Q:", q)
    print("   raw answers        :", raw)
    print("   grouped by meaning :", dict(Counter(canon(s) for s in raw)))
    print(f"   majority idea      : {top!r}   agreement: {conf:.0%}")
    print(f"   action             : {'ANSWER' if conf>=0.7 else 'ESCALATE to a clinician'}\n")


### The confidence bar is a dial — watch it move, per question
For each question we sample the model, group its answers by meaning, and get an **agreement %**. Then we sweep the bar (`tau`) from lax (0.5) to strict (0.9). Reading **down** a column shows the trade-off: raise the bar and more questions flip from **ANS** (answered) to **esc** (escalated to a clinician). Factual lookups sit near 100% and stay answered; open-ended questions sit lower and get escalated first — which is exactly what you want.


In [ ]:
# One row per question: its agreement, then ANSWER/escalate at each threshold (tau).
QS = [
 ("creatinine  (factual)",    "What is the most recent creatinine? (value only)",  "1.2 mg/dL"),
 ("potassium   (factual)",    "What is the most recent potassium? (value only)",   "4.1 mmol/L"),
 ("warfarin    (factual)",    "What is the current warfarin dose? (value only)",   "5 mg"),
 ("chest-pain cause (open)",  "Single most likely cause of the chest pain? (one phrase)", None),
 ("hypotension cause (open)", "Single most likely cause of the transient hypotension? (one phrase)", None),
]
N, TAUS = 12, [0.5, 0.6, 0.7, 0.8, 0.9]

hdr = f"{'question':28}{'agree':>6}   " + "  ".join(f"tau {t:>3}" for t in TAUS)
print(hdr); print("-" * len(hdr))
for label, q, correct in QS:
    top, conf, _ = confidence(q, N)
    grid = "  ".join(f"{'ANS' if conf>=t else 'esc':>6}" for t in TAUS)
    tag = "" if correct is None else ("   [correct]" if matches(top, correct) else "   [WRONG]")
    print(f"{label:28}{conf:>5.0%}   {grid}{tag}")
print("\nANS = answered  ·  esc = escalated to a clinician.")
print("Read DOWN a tau column: raising the bar turns more answers into escalations.")
print("Factual lookups sit near 100% and stay ANSWERED; open questions sit lower and flip to esc first.")
print("Agreement = consistency, NOT truth — a high-agreement answer to an open question can be confidently wrong.")


### Takeaway
Abstention turns a silent wrong answer into a routed one. As the threshold rises, escalations rise — that trade-off *is* the design decision, and it belongs to the clinical workflow, not the model. Grounding the questions in the record is what makes the confidence meaningful: the model is confident on what it was given and unsure on what it wasn't. Better uncertainty signals (calibrated probabilities, conformal prediction, semantic entropy) are exactly where a research program can add value.

*Try:* with a real model set, re-run — the spread now comes from the model itself; watch which questions it is quietly unsure about, and remember agreement is not the same as being right.


## Exercise

No programming is required for this section.

**Activity:** Choose an escalation threshold for a low-consequence lookup and a high-consequence recommendation. Explain why the thresholds may differ and what the receiving clinician needs to see.

### Discussion prompts

- Can several model samples agree on the same wrong answer?
- Who bears the workload created by abstention?
- Should confidence thresholds vary by deployment rung?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: change the threshold and compare the workflow consequence.
AGREEMENT = 0.72
ABSTENTION_THRESHOLD = 0.80
CONSEQUENCE = "high"  # try "low"
action = "answer" if AGREEMENT >= ABSTENTION_THRESHOLD else "escalate"
print(f"Agreement={AGREEMENT:.0%}; threshold={ABSTENTION_THRESHOLD:.0%}; action={action.upper()}")
print("Reminder: agreement measures consistency, not truth.")


## Optional technical extension

Add a labeled evaluation set and calculate coverage, selective accuracy, error among answered cases, and escalation rate. Compare agreement with a calibrated confidence method.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

Do not describe agreement as a clinically calibrated probability. The exercise is about selective prediction and workflow routing; validation requires labeled cases and consequence-aware thresholds.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
